# Unit 1 — decision tree, KNN, logistic regression

Phase 5. Diagram-type classification from 33 handcrafted features (Phase 4), under cross-validation grouped by scribe so that no writer appears in both train and test.

This notebook reads committed results from `reports/`. It reproduces nothing and needs no corpus or GPU. `reports/README.md` names the command that regenerates each file. The full syllabus mapping is in `docs/syllabus_map.md`.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "reports").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CELLS = json.loads((ROOT / "reports" / "master_results.json").read_text(encoding="utf-8"))["cells"]


def results(*stages):
    """The master table's rows for these stages, as a Markdown table."""
    rows = ["| stage | model | metric | value | target | verdict |", "| :-- | :-- | :-- | --: | --: | :-: |"]
    for c in CELLS:
        if c["stage"] in stages and c.get("available", True):
            verdict = "" if c.get("passes") is None else ("pass" if c["passes"] else "**fail**")
            target = "" if c.get("target") is None else c["target"]
            rows.append(f"| {c['stage']} | {c['model']} | {c['metric']} | {c['value']} | {target} | {verdict} |")
    display(Markdown("\n".join(rows)))


def report(name, *keys):
    """Selected top-level fields of one reports/*.json."""
    data = json.loads((ROOT / "reports" / name).read_text(encoding="utf-8"))
    return {k: data[k] for k in keys if k in data}


## The three classifiers (5.1)

Macro F1 against a majority baseline. Source: `reports/classification_report.md`.

In [2]:
results("classify")

| stage | model | metric | value | target | verdict |
| :-- | :-- | :-- | --: | --: | :-: |
| classify | CLIP ViT-B/32 + OvO RBF SVM | accuracy | 0.9871 | 0.92 | pass |
| classify | CLIP ViT-B/32 + OvO RBF SVM | accuracy std | 0.0009 |  |  |
| classify | CLIP ViT-B/32 + OvO RBF SVM | worst repeat accuracy | 0.9858 |  |  |
| classify | CLIP ViT-B/32 + OvO RBF SVM | macro F1 | 0.9518 |  |  |
| classify | handcrafted logreg | macro F1 | 0.7917 |  |  |
| classify | handcrafted logreg | balanced accuracy | 0.7891 |  |  |
| classify | handcrafted knn | macro F1 | 0.7614 |  |  |
| classify | handcrafted knn | balanced accuracy | 0.7523 |  |  |
| classify | handcrafted tree | macro F1 | 0.7434 |  |  |
| classify | handcrafted tree | balanced accuracy | 0.7596 |  |  |
| classify | majority baseline | macro F1 | 0.1237 |  |  |
| classify | NB pipeline router | test accuracy | 1.0 |  |  |
| classify | handcrafted logreg (14.2 rerun) | macro F1 | 0.9769 |  |  |
| classify | embedding SVM (14.2 rerun) | macro F1 | 1.0 |  |  |

![p5_tree.png](../reports/figures/p5_tree.png)

`reports/figures/p5_tree.png`

## Precision, recall, ROC and calibration (5.2)

![p5_pr.png](../reports/figures/p5_pr.png)

`reports/figures/p5_pr.png`

![p5_roc.png](../reports/figures/p5_roc.png)

`reports/figures/p5_roc.png`

![p5_calibration.png](../reports/figures/p5_calibration.png)

`reports/figures/p5_calibration.png`

![p5_confusion.png](../reports/figures/p5_confusion.png)

`reports/figures/p5_confusion.png`

![p5_learning_curves.png](../reports/figures/p5_learning_curves.png)

`reports/figures/p5_learning_curves.png`

## KNN decision boundaries (5.3.1)

Projecting to two dimensions costs half the model: the best projected macro F1 is 0.583, against 0.764 on all 33 features. The figure shows where the classes sit, not how the model separates them.

![p5_knn_boundaries.png](../reports/figures/p5_knn_boundaries.png)

`reports/figures/p5_knn_boundaries.png`